In [ ]:
#!/usr/bin/env python3
"""
==============================================================================
HAOR FLOOD — MULTI-YEAR ANALYSIS  (haor_flood_multiyear.ipynb)
==============================================================================
One notebook that runs the WHOLE per-year cluster analysis for every year, then
compares across years. It reuses the exact logic from your two single-year
notebooks (extract_onset_masked, per-haor flooded-fraction hydrographs,
study-area-restricted dynamic connectivity, per-pixel onset/offset), wrapped in
a function that takes a year and returns that year's results.

INPUT (from haor_export_multiyear.py, downloaded to the multiyear folder):
  flood_stack_cluster_<year>.tif   first_wet_date_<year>.tif   last_wet_date_<year>.tif
  + haors_manual.shp (your 5 digitized haors)  + a JRC occurrence raster

WHAT IT PRODUCES, per year and then pooled:
  PER YEAR (written to analysis/<year>/):
    - haor_hydrograph_curves_<year>.csv      (per-haor flooded fraction over time)
    - haor_hydrograph_timing_<year>.csv      (onset/peak/recession per haor)
    - connectivity_curve_<year>.csv          (merge/fragment + largest_frac)
    - haor_connection_matrix_<year>.csv      (which component each haor is in, per date)
    - haor_onset_perhaor_<year>.csv          (per-pixel onset, permanent water removed)
    - haor_offset_perhaor_<year>.csv         (per-pixel offset / dry-down)
    - figures: hydrographs, connectivity, onset+offset boxplots
  CROSS-YEAR (the multi-year payoff):
    - multiyear_summary.csv  (one row per year: onset/recession spreads, merge stats)
    - figure: onset-spread vs recession-spread across years (synchrony stability)
    - figure: per-haor onset & offset medians across years (does ordering repeat?)
    - figure: connectivity "fraction of season fully merged" per year

The multi-year QUESTION: do synchronous filling, the merge-and-fragment arc, and
the staggered recession (Gumaria-first) REPEAT every year, or was 2025 a one-off?
A pattern stable across 2019-2025 turns a single observation into a finding.
==============================================================================
"""
import os, re, datetime, glob
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.features import geometry_mask
from rasterio.mask import mask as rmask
from rasterio.warp import reproject, Resampling
from shapely.geometry import mapping
from scipy import ndimage
print("\u2713 imports ok")


In [ ]:
# %%
# ============================ PATHS & CONFIG ============================
# Point BASE at the multiyear folder you downloaded from Drive.
BASE      = "/work/a06/wasif/haor_flood_analysis_multiyear"
HAORS_SHP = "/work/a06/wasif/haor_flood_analysis/digitization/haors_manual.shp"
# JRC occurrence (cluster version preferred; falls back to the broad one)
JRC_CANDIDATES = [
    "/work/a06/wasif/haor_flood_analysis/haor_digitization_tanguar/tanguar_jrc_occurrence.tif",
    "/work/a06/wasif/haor_flood_analysis/jrc_water_occurrence.tif",
    os.path.join(BASE, "tanguar_jrc_occurrence.tif"),
]
JRC_TIF = next((p for p in JRC_CANDIDATES if os.path.exists(p)), None)
assert JRC_TIF, "No JRC occurrence raster found - check JRC_CANDIDATES paths."

ANALYSIS_ROOT = os.path.join(BASE, "analysis")
os.makedirs(ANALYSIS_ROOT, exist_ok=True)

# analysis knobs (identical to your single-year notebooks)
PERMANENT_WATER_THRESHOLD = 75     # JRC occurrence % above which = permanent water
LEVEL = 0.50                        # onset/recession = 50% of each haor's seasonal max
MIN_PX, BRIDGE = 10, 1             # connectivity speckle + 1-px bridge guard
MIN_OBSERVED_FRAC = 0.50           # need >=50% of cluster observed to score a date
BUFFER_PX = 8                      # ~800 m study-area ring around the 5 haors
CENTRAL_HAOR = "GMA"               # Gumaria = central connector

# which years are present in the folder (auto-detected from the stacks)
YEARS = sorted(int(re.search(r"(\d{4})", os.path.basename(f)).group(1))
               for f in glob.glob(os.path.join(BASE, "flood_stack_cluster_*.tif")))
print("JRC:", JRC_TIF)
print("years found:", YEARS)

os.environ["SHAPE_RESTORE_SHX"] = "YES"
haors = gpd.read_file(HAORS_SHP)
haors = haors[haors["haor_id"] != "DUMMY"].copy()
print("haors:", haors["haor_id"].tolist())


In [ ]:
# %%
# ============================ REUSED HELPER ============================
# Verbatim from 5_haor_flood_onset2.ipynb. Works for BOTH onset and offset
# rasters (it just extracts per-pixel day values inside a haor, minus permanent
# water), so we call it for first_wet AND last_wet.
def extract_onset_masked(haor_geom, fwd_src, jrc_src):
    geom_fwd = gpd.GeoSeries([haor_geom], crs=haors.crs).to_crs(fwd_src.crs).iloc[0]
    fwd_data, fwd_transform = rmask(fwd_src, [mapping(geom_fwd)], crop=True, filled=False)
    fwd_arr = fwd_data[0]
    fwd_mask = np.ma.getmaskarray(fwd_arr) if np.ma.isMaskedArray(fwd_arr) else np.zeros(fwd_arr.shape, bool)
    fwd_vals = np.ma.filled(fwd_arr, np.nan).astype("float32")
    out_h, out_w = fwd_vals.shape
    jrc_on_grid = np.full((out_h, out_w), np.nan, dtype="float32")
    reproject(source=rasterio.band(jrc_src, 1), destination=jrc_on_grid,
              src_transform=jrc_src.transform, src_crs=jrc_src.crs,
              dst_transform=fwd_transform, dst_crs=fwd_src.crs,
              resampling=Resampling.bilinear)
    valid = (~fwd_mask) & np.isfinite(fwd_vals) & (fwd_vals > 0)
    not_perm = ~(np.isfinite(jrc_on_grid) & (jrc_on_grid > PERMANENT_WATER_THRESHOLD))
    keep = valid & not_perm
    return fwd_vals[keep], int(np.sum(valid & ~not_perm))

def _doy(desc):
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", desc or "")
    if not m: return np.nan
    y, mo, d = map(int, m.groups())
    return datetime.date(y, mo, d).timetuple().tm_yday
print("\u2713 helpers defined")


In [ ]:
# %%
# ====================== PER-YEAR ANALYSIS FUNCTION ======================
# Runs the full single-year pipeline for ONE year and returns a results dict.
# This is your part4 hydrograph + connectivity and your onset2 onset/offset,
# refactored to take a year so we can loop. Figures are saved per year.
def analyze_year(year, make_figs=True):
    stack_tif = os.path.join(BASE, f"flood_stack_cluster_{year}.tif")
    fwd_tif   = os.path.join(BASE, f"first_wet_date_{year}.tif")
    lwd_tif   = os.path.join(BASE, f"last_wet_date_{year}.tif")
    for p in (stack_tif, fwd_tif, lwd_tif):
        if not os.path.exists(p):
            print(f"  [{year}] missing {os.path.basename(p)} - skipping year"); return None
    outdir = os.path.join(ANALYSIS_ROOT, str(year)); os.makedirs(outdir, exist_ok=True)

    # ---- read the composited stack ----
    with rasterio.open(stack_tif) as src:
        nod = src.nodata
        hv = haors.to_crs(src.crs)
        days = np.array([_doy(d) for d in src.descriptions], float)
        bands = []
        for i in range(src.count):
            b = src.read(i + 1).astype("float32")
            if nod is not None: b[b == nod] = np.nan
            bands.append(b)
        shape, transform = (src.height, src.width), src.transform
        masks = {h.haor_id: geometry_mask([h.geometry], shape, transform, invert=True)
                 for _, h in hv.iterrows()}

    # ---- per-haor flooded-fraction hydrographs (flooded / observed) ----
    curves = {}
    for hid, m in masks.items():
        fr = []
        for b in bands:
            sub = b[m]; obs = np.isfinite(sub); nobs = int(obs.sum())
            fr.append(int((sub[obs] == 1).sum()) / nobs if nobs else np.nan)
        curves[hid] = np.array(fr, float)
    curves_df = pd.DataFrame({"doy": days, **curves}).sort_values("doy")
    curves_df.to_csv(os.path.join(outdir, f"haor_hydrograph_curves_{year}.csv"), index=False)

    # ---- onset / peak / recession from each curve ----
    trows = []
    for hid in curves:
        f = curves[hid]; ok = np.isfinite(f)
        d, ff = days[ok], f[ok]; o = np.argsort(d); d, ff = d[o], ff[o]
        if ff.size == 0 or not np.isfinite(np.nanmax(ff)): continue
        fmax = np.nanmax(ff); above = d[ff >= LEVEL * fmax]
        trows.append(dict(haor_id=hid,
            onset_hydro=float(above.min()) if above.size else np.nan,
            peak_hydro=float(d[np.argmax(ff)]),
            recession_hydro=float(above.max()) if above.size else np.nan,
            max_fraction=round(float(fmax), 3)))
    timing = pd.DataFrame(trows)
    timing.to_csv(os.path.join(outdir, f"haor_hydrograph_timing_{year}.csv"), index=False)

    # ---- dynamic connectivity (study-area restricted) ----
    cluster_pixels = np.zeros(shape, bool)
    for m in masks.values(): cluster_pixels |= m
    study = ndimage.binary_dilation(cluster_pixels, iterations=BUFFER_PX)
    n_cluster = int(cluster_pixels.sum())
    crecs, cmat = [], []
    for i, b in enumerate(bands):
        observed = np.isfinite(b)
        obs_frac = (observed & cluster_pixels).sum() / n_cluster if n_cluster else 0.0
        if obs_frac < MIN_OBSERVED_FRAC:
            crecs.append(dict(doy=days[i], n_components=-1, largest_frac=-1, total_wet=-1,
                              obs_frac=round(obs_frac, 3), all_connected=False))
            cmat.append(dict(doy=days[i], **{h: -1 for h in masks})); continue
        water = observed & (b == 1) & study
        if BRIDGE: water = ndimage.binary_opening(water, iterations=BRIDGE)
        lab, n = ndimage.label(water)
        if n:
            sz = ndimage.sum(np.ones_like(lab), lab, index=np.arange(1, n + 1))
            water &= np.isin(lab, np.where(sz >= MIN_PX)[0] + 1)
            lab, n = ndimage.label(water)
        total = int(water.sum())
        largest = float(ndimage.sum(np.ones_like(lab), lab,
                        index=np.arange(1, n + 1)).max() / total) if (n and total) else 0.0
        comp = {}
        for hid, m in masks.items():
            v = lab[m & (lab > 0)]; comp[hid] = int(np.bincount(v).argmax()) if v.size else 0
        nz = [x for x in comp.values() if x > 0]
        crecs.append(dict(doy=days[i], n_components=int(n), largest_frac=round(largest, 3),
                          total_wet=total, obs_frac=round(obs_frac, 3),
                          all_connected=(len(nz) == len(masks) and len(set(nz)) == 1)))
        cmat.append(dict(doy=days[i], **comp))
    conn_df = pd.DataFrame(crecs).sort_values("doy")
    matrix_df = pd.DataFrame(cmat).sort_values("doy")
    conn_df.to_csv(os.path.join(outdir, f"connectivity_curve_{year}.csv"), index=False)
    matrix_df.to_csv(os.path.join(outdir, f"haor_connection_matrix_{year}.csv"), index=False)

    # ---- per-pixel onset & offset (permanent water removed) ----
    def perpixel(tif, tag):
        rows, dist = [], {}
        with rasterio.open(tif) as src_r, rasterio.open(JRC_TIF) as jsrc:
            for _, h in haors.iterrows():
                kept, ndrop = extract_onset_masked(h.geometry, src_r, jsrc)
                dist[h.haor_id] = kept
                if kept.size:
                    rows.append(dict(haor_id=h.haor_id, n_pixels=kept.size,
                        **{f"{tag}_p10": np.percentile(kept, 10),
                           f"{tag}_median": np.median(kept),
                           f"{tag}_p90": np.percentile(kept, 90),
                           f"{tag}_std": np.std(kept)}))
        return pd.DataFrame(rows), dist
    onset_df, onset_dist = perpixel(fwd_tif, "onset")
    offset_df, offset_dist = perpixel(lwd_tif, "offset")
    onset_df.to_csv(os.path.join(outdir, f"haor_onset_perhaor_{year}.csv"), index=False)
    offset_df.to_csv(os.path.join(outdir, f"haor_offset_perhaor_{year}.csv"), index=False)

    # ---- per-year figures ----
    if make_figs:
        # hydrographs
        o = np.argsort(days)
        plt.figure(figsize=(11, 4))
        for hid, fr in curves.items(): plt.plot(days[o], fr[o], marker="o", ms=2, label=hid)
        plt.title(f"Per-haor flooded fraction {year}"); plt.xlabel("day of year")
        plt.ylabel("share underwater"); plt.legend(fontsize=8); plt.grid(alpha=.3)
        plt.tight_layout(); plt.savefig(os.path.join(outdir, f"hydrographs_{year}.png"), dpi=150); plt.close()
        # connectivity
        valid = conn_df[conn_df["n_components"] >= 0]
        central = CENTRAL_HAOR if CENTRAL_HAOR in masks else list(masks)[0]
        mvalid = matrix_df[matrix_df[central] >= 0]
        linked = [int(sum(r[h] == r[central] and r[central] > 0 for h in masks)) for _, r in mvalid.iterrows()]
        fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
        ax[0].plot(valid["doy"], valid["largest_frac"], "o-", ms=3, color="#1C7293")
        ax[0].set_ylabel("largest / study water"); ax[0].set_title(f"Connectivity {year}"); ax[0].grid(alpha=.3)
        ax[1].plot(mvalid["doy"], linked, "o-", ms=3, color="#933556")
        ax[1].set_ylabel(f"# haors w/ {central}"); ax[1].set_xlabel("day of year"); ax[1].grid(alpha=.3)
        plt.tight_layout(); plt.savefig(os.path.join(outdir, f"connectivity_{year}.png"), dpi=150); plt.close()
        # onset + offset boxplots side by side
        fig, ax = plt.subplots(1, 2, figsize=(14, 5))
        oo = onset_df.sort_values("onset_median")["haor_id"].tolist()
        ax[0].boxplot([onset_dist[h] for h in oo], tick_labels=oo, showfliers=False)
        ax[0].set_title(f"Onset {year}"); ax[0].set_ylabel("day of year")
        ff = offset_df.sort_values("offset_median")["haor_id"].tolist()
        ax[1].boxplot([offset_dist[h] for h in ff], tick_labels=ff, showfliers=False)
        ax[1].set_title(f"Offset {year}"); ax[1].set_ylabel("day of year")
        plt.tight_layout(); plt.savefig(os.path.join(outdir, f"onset_offset_box_{year}.png"), dpi=150); plt.close()

    # ---- one-row summary for the cross-year table ----
    onset_spread = float(timing["onset_hydro"].max() - timing["onset_hydro"].min())
    rec_spread   = float(timing["recession_hydro"].max() - timing["recession_hydro"].min())
    valid = conn_df[conn_df["n_components"] >= 0]
    frac_full_merge = float((valid["all_connected"]).mean()) if len(valid) else np.nan
    summary = dict(year=year,
        onset_spread=onset_spread, recession_spread=rec_spread,
        onset_within_std=float(onset_df["onset_std"].mean()),
        offset_within_std=float(offset_df["offset_std"].mean()),
        offset_between_spread=float(offset_df["offset_median"].max() - offset_df["offset_median"].min()),
        frac_dates_all5_merged=round(frac_full_merge, 3),
        earliest_drain=offset_df.sort_values("offset_median")["haor_id"].iloc[0],
        latest_drain=offset_df.sort_values("offset_median")["haor_id"].iloc[-1])
    return dict(summary=summary, timing=timing, onset_df=onset_df, offset_df=offset_df,
                conn_df=conn_df, matrix_df=matrix_df)
print("\u2713 analyze_year defined")


In [ ]:
# %%
# ============================ RUN ALL YEARS ============================
results = {}
for y in YEARS:
    print(f"--- analyzing {y} ---")
    r = analyze_year(y, make_figs=True)
    if r is not None:
        results[y] = r
        s = r["summary"]
        print(f"  onset_spread={s['onset_spread']:.0f}d  rec_spread={s['recession_spread']:.0f}d  "
              f"merged_frac={s['frac_dates_all5_merged']}  drain {s['earliest_drain']}->{s['latest_drain']}")

summary_df = pd.DataFrame([results[y]["summary"] for y in results]).sort_values("year")
summary_df.to_csv(os.path.join(ANALYSIS_ROOT, "multiyear_summary.csv"), index=False)
print("\n=== MULTI-YEAR SUMMARY ===")
print(summary_df.to_string(index=False))


In [ ]:
# %%
# ===================== CROSS-YEAR FIGURE 1: synchrony stability =====================
# Onset spread vs recession spread per year. The 2025 story was: filling
# synchronous (small onset spread), draining more staggered (bigger recession
# spread). Does that hold every year?
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(summary_df["year"], summary_df["onset_spread"], "o-", label="onset spread (fill)")
ax.plot(summary_df["year"], summary_df["recession_spread"], "s-", label="recession spread (drain)")
ax.plot(summary_df["year"], summary_df["onset_within_std"], "--", color="grey", label="within-haor scatter (onset)")
ax.set_xlabel("year"); ax.set_ylabel("days between haors")
ax.set_title("Synchrony stability: do haors fill together but drain apart every year?")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(ANALYSIS_ROOT, "multiyear_synchrony.png"), dpi=150); plt.show()


In [ ]:
# %%
# ===================== CROSS-YEAR FIGURE 2: per-haor timing across years =====================
# Does the ORDER repeat? e.g. is Gumaria consistently the earliest to drain?
onset_med = {}; offset_med = {}
for y in results:
    onset_med[y] = results[y]["onset_df"].set_index("haor_id")["onset_median"]
    offset_med[y] = results[y]["offset_df"].set_index("haor_id")["offset_median"]
onset_M = pd.DataFrame(onset_med)    # rows=haor, cols=year
offset_M = pd.DataFrame(offset_med)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for hid in onset_M.index:
    ax[0].plot(onset_M.columns, onset_M.loc[hid], "o-", label=hid)
ax[0].set_title("Onset median per haor across years"); ax[0].set_ylabel("day of year"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
for hid in offset_M.index:
    ax[1].plot(offset_M.columns, offset_M.loc[hid], "o-", label=hid)
ax[1].set_title("Offset (drain) median per haor across years"); ax[1].set_ylabel("day of year"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(ANALYSIS_ROOT, "multiyear_perhaor_timing.png"), dpi=150); plt.show()

print("Drain order each year (earliest -> latest):")
for y in results:
    od = results[y]["offset_df"].sort_values("offset_median")["haor_id"].tolist()
    print(f"  {y}: {' < '.join(od)}")


In [ ]:
# %%
# ===================== CROSS-YEAR FIGURE 3: how merged is each year =====================
# Fraction of observed dates where all 5 haors are one body. High + stable =
# the merge is a reliable seasonal property, not a 2025 fluke.
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(summary_df["year"].astype(str), summary_df["frac_dates_all5_merged"], color="#1C7293")
ax.set_ylabel("fraction of observed dates with all 5 haors merged")
ax.set_title("How often do the 5 haors form ONE body, by year?")
ax.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.savefig(os.path.join(ANALYSIS_ROOT, "multiyear_merge_fraction.png"), dpi=150); plt.show()

print("\nInterpretation guide:")
print(" - onset spread small & stable every year  -> synchronous filling is a robust property")
print(" - recession spread consistently > onset    -> staggered drainage is real, not 2025-only")
print(" - same haor drains first across years       -> structural (outlet/elevation), worth the barrier analysis")
print(" - merge fraction high & stable              -> the haors genuinely act as one body at high water")
